<a href="https://colab.research.google.com/github/salsabila-fadhilah/ISYS2001---Salsabila-Fadhilah/blob/main/Module%2005%20-%20Function%20Junction/Activity_2_Enhancing_Your_Calculator_with_pyinputplus.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/kevin-blasiak-curtin/ISYS2001-Archive/blob/main/Module%2005%20-%20Function%20Junction/Activity_2_Enhancing_Your_Calculator_with_pyinputplus.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Introduction

Your calculator from Activity 1 works, but only when the user does exactly what you expect. Type a letter where it asks for a number and the programme stops with an error. Type 9 at the menu and it quietly does nothing useful. Real users make these mistakes constantly, so a programme that assumes perfect input is not finished.

In this worksheet you will make the calculator robust: it should keep asking until it gets input it can use, rather than crashing. You will do this with the validation pattern you met in Week 4, a `while` loop wrapped around a `try`/`except` block. No new libraries are needed; you already have the tools.

We will also stop to think about a decision you will face often as an information systems professional: when a problem like this comes up, do you write the solution yourself, or reach for an existing library that does it for you? That judgement matters as much as the code.

**By the end of this worksheet you should be able to:**

- recognise where a programme is fragile and predict how it will fail
- use a `while` loop with `try`/`except` to validate user input
- keep your modular design intact while adding this behaviour
- reason about when taking on an external library is worth it, and when it is not

# Seeing the problem first hand

Before fixing anything, it helps to see the failure yourself. The cell below is the calculator you built in Activity 1. Run it, and at the "first number" prompt type a word such as `abc` instead of a number.

You should see the programme stop with a `ValueError`. Try it again and enter `9` at the menu instead of 1 to 4, and notice that the result is unhelpful rather than a clear message. These are the two weaknesses we will address.

In [1]:
# The calculator from Activity 1 (our starting point)

def add(a, b):
    return a + b

def subtract(a, b):
    return a - b

def multiply(a, b):
    return a * b

def divide(a, b):
    if b == 0:
        return "Error: Cannot divide by zero."
    else:
        return a / b

def get_operation():
    print("Simple Calculator Menu")
    print("1. Add")
    print("2. Subtract")
    print("3. Multiply")
    print("4. Divide")
    choice = input("Enter your choice (1-4): ")
    return choice

def get_number(prompt):
    return float(input(prompt))

def perform_calculation(operation, num1, num2):
    if operation == "1":
        return add(num1, num2)
    elif operation == "2":
        return subtract(num1, num2)
    elif operation == "3":
        return multiply(num1, num2)
    elif operation == "4":
        return divide(num1, num2)
    else:
        return "Invalid choice."

# Run the calculator and try to break it with bad input
operation = get_operation()
num1 = get_number("Enter the first number: ")
num2 = get_number("Enter the second number: ")
result = perform_calculation(operation, num1, num2)
print("Result:", result)

Simple Calculator Menu
1. Add
2. Subtract
3. Multiply
4. Divide
Enter your choice (1-4): 4
Enter the first number: 1
Enter the second number: 2
Result: 0.5


# How should we handle bad input?

Once you have seen the problem, the professional habit is to pause and ask how the problem is usually solved, rather than jumping straight to code. This is a good moment to use AI as a research aid. A prompt along these lines works well:

```
I am building a small Python calculator. When the user types text instead
of a number, the programme crashes with a ValueError. What are the common
ways to handle invalid user input in Python, and what are the trade-offs
of each? Assume I am a beginner using only standard Python.
```

You will usually see two broad routes.

**Route one: handle it yourself.** Wrap the input in a loop that keeps asking until the value is acceptable, using `try`/`except` to catch the error. This is exactly the pattern from Week 4. It needs no extra software, you understand every line, and you keep full control over the messages the user sees.

**Route two: use a library.** There are libraries, such as `pyinputplus`, written specifically to validate input for you. They can save typing and handle edge cases you might not think of.

So which do you choose? This is a genuine judgement, not an obvious win either way. A library is another dependency: something extra to install, that can stop being maintained, and that you understand less well than code you wrote. For a small, well-understood problem that you can solve in a few lines with tools you already have, writing it yourself is usually the better call, and it is the one we will take here. Later in your degree, when a problem is large or genuinely hard to get right, the balance tips the other way. Being able to weigh that up is part of the job.

Note that the constraint "beginner, standard Python only" in the prompt above did real work: it stopped the AI steering you toward a library before you had thought about whether you wanted one. Keep giving your prompts that kind of specific direction.

# Step 1: make get_number robust

We want `get_number()` to keep asking until it receives something it can turn into a number, instead of stopping the programme.

**Pseudo-code:**

```
function get_number(prompt):
    repeat:
        read the user's input
        try to convert it to a float
            if it works, return the number
        if it fails:
            tell the user and ask again
```

Before you look at the code below, try writing your own prompt to an AI that describes this behaviour precisely. Name the function, say it should loop until the input is valid, and say it should use `try`/`except`. Compare what you get with the version here.

**Code:**

```python
def get_number(prompt):
    while True:
        try:
            return float(input(prompt))
        except ValueError:
            print("That is not a number. Please try again.")
```

The `return` inside the loop is what ends it: once we have a valid number, we hand it back and the loop stops. If `float()` fails, the `except` block runs, we print a message, and the loop goes round again.

In [2]:
def get_number(prompt):
    while True:
        try:
            return float(input(prompt))
        except ValueError:
            print("That is not a number. Please try again.")

# Try it: run this and type some text before entering a real number
value = get_number("Enter a number: ")
print("You entered:", value)

Enter a number: 4
You entered: 4.0


# Step 2: make get_operation robust

The menu has the same weakness. If the user types 9, or presses enter by mistake, the calculator should ask again rather than accept it. We can loop until the choice is one of the four we allow.

**Pseudo-code:**

```
function get_operation():
    repeat:
        display the menu
        read the user's choice
        if the choice is one of "1", "2", "3", "4":
            return it
        otherwise:
            tell the user and ask again
```

**Code:**

```python
def get_operation():
    valid_choices = ["1", "2", "3", "4"]
    while True:
        print("Simple Calculator Menu")
        print("1. Add")
        print("2. Subtract")
        print("3. Multiply")
        print("4. Divide")
        choice = input("Enter your choice (1-4): ")
        if choice in valid_choices:
            return choice
        print("Please choose 1, 2, 3 or 4.")
```

Because `get_operation()` now only ever returns a valid choice, the `else: "Invalid choice."` branch in `perform_calculation()` becomes a safety net that should never fire. That is fine; leaving it in is sensible defensive programming.

In [3]:
def get_operation():
    valid_choices = ["1", "2", "3", "4"]
    while True:
        print("Simple Calculator Menu")
        print("1. Add")
        print("2. Subtract")
        print("3. Multiply")
        print("4. Divide")
        choice = input("Enter your choice (1-4): ")
        if choice in valid_choices:
            return choice
        print("Please choose 1, 2, 3 or 4.")

# Try it: run this and enter something outside 1-4 before choosing
op = get_operation()
print("You chose:", op)

Simple Calculator Menu
1. Add
2. Subtract
3. Multiply
4. Divide
Enter your choice (1-4): 3
You chose: 3


# Putting it together

Notice what did not change. The calculation functions (`add`, `subtract`, `multiply`, `divide`) and `perform_calculation()` are untouched. We only strengthened the two functions that deal with the user. That is the payoff of modular design: because each function has one clear job, we could improve input handling without disturbing anything else.

Run the cell below. It defines the full, robust calculator and runs it. Try to break it with bad input, as you did at the start, and confirm that it now recovers instead of crashing.

In [4]:
# The full, robust calculator

def add(a, b):
    return a + b

def subtract(a, b):
    return a - b

def multiply(a, b):
    return a * b

def divide(a, b):
    if b == 0:
        return "Error: Cannot divide by zero."
    else:
        return a / b

def get_operation():
    valid_choices = ["1", "2", "3", "4"]
    while True:
        print("Simple Calculator Menu")
        print("1. Add")
        print("2. Subtract")
        print("3. Multiply")
        print("4. Divide")
        choice = input("Enter your choice (1-4): ")
        if choice in valid_choices:
            return choice
        print("Please choose 1, 2, 3 or 4.")

def get_number(prompt):
    while True:
        try:
            return float(input(prompt))
        except ValueError:
            print("That is not a number. Please try again.")

def perform_calculation(operation, num1, num2):
    if operation == "1":
        return add(num1, num2)
    elif operation == "2":
        return subtract(num1, num2)
    elif operation == "3":
        return multiply(num1, num2)
    elif operation == "4":
        return divide(num1, num2)
    else:
        return "Invalid choice."

# Main script
operation = get_operation()
num1 = get_number("Enter the first number: ")
num2 = get_number("Enter the second number: ")
result = perform_calculation(operation, num1, num2)
print("Result:", result)

Simple Calculator Menu
1. Add
2. Subtract
3. Multiply
4. Divide
Enter your choice (1-4): 4
Enter the first number: 1
Enter the second number: 2
Result: 0.5


# Weighing it up

You have now solved the problem with about a dozen lines and no extra software. It is worth being honest about the trade-off you made.

Writing the validation yourself gave you full control and full understanding, no dependency to install or maintain, and good practice with a pattern you will reuse constantly. The cost is that you had to write and test it, and a hand-written check can miss an edge case that a mature library would have covered.

A library such as `pyinputplus` would have done the same job in fewer lines and handled some cases for you. The cost is a dependency your code now relies on, less control over the exact behaviour, and one more thing that can break or fall out of maintenance.

For a problem this size, with tools you already understand, doing it yourself is the reasonable choice, and it is the habit we want in this unit: reach for the simplest thing that solves the problem well. The point is not that libraries are bad. It is that adding one is a decision with costs, and you should make it deliberately rather than by reflex. That habit, knowing what a tool costs before you adopt it, is exactly the kind of judgement an information systems graduate is expected to bring.

# Optional challenges

If you have time, extend the robust calculator. Try to write your own detailed prompt for each, naming the behaviour you want, before asking an AI for help.

1. **Reject a zero divisor at input.** At the moment `divide()` reports the error after the fact. Change the flow so that, when the user chooses divide, the second number is re-requested until it is not zero.
2. **Add a way to quit.** Let the user type `q` at the menu to exit cleanly instead of always doing one calculation.
3. **Limit the retries.** Change `get_number()` so that after, say, three failed attempts it gives up and returns a default, rather than looping forever. Think about whether that is actually a good idea for this programme, and note your reasoning.

# Reflection

Like Activity 1, this worksheet is practice. You do not submit it on its own; it prepares you for this week's lab exit ticket, which is the finance mini-project. Keep your notes, since you may be asked to talk through your reasoning.

For your own learning, write a short reflection (around 200 to 300 words) on:

1. **Robustness:** which parts of the original calculator were fragile, and how did the `while` and `try`/`except` pattern fix them?
2. **Modular design:** why were you able to change the input handling without touching the calculation functions?
3. **Judgement:** how did you decide between writing the validation yourself and using a library? What would change your answer for a larger problem?
4. **Working with AI:** were your prompts specific enough to get useful answers? What would you phrase differently next time?

You have finished this worksheet if your calculator no longer crashes on bad input, you can explain how the validation loop works, and you can give a reasoned answer to when you would and would not add an external library.

Next you will apply all of this in the finance mini-project, where you will build a small set of modular functions of your own.